In [ ]:
"""
LAB SESSION 2 — PHASE 1 : DATA COLLECTION
Compatible Google Colab
"""
!pip install yfinance ta -q
import logging
import yfinance as yf
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator

# ─────────────────────────────────────────────
# Dossier de sauvegarde dans Google Colab
# ─────────────────────────────────────────────
RAW_DATA_DIR = Path("/content/data/raw")

# ─────────────────────────────────────────────
# Paires Forex
# ─────────────────────────────────────────────
FOREX_PAIRS = {
    "EURUSD": "EURUSD=X",
    "EURMAD": "EURMAD=X",
    "USDMAD": "USDMAD=X",
}

START_DATE = "2015-01-01"
END_DATE   = "2025-12-31"

# ─────────────────────────────────────────────
# Logging
# ─────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────
# Indicateurs techniques
# ─────────────────────────────────────────────
def add_indicators(df):

    # Moyennes mobiles
    df["SMA_10"] = SMAIndicator(
        close=df["Close"],
        window=10
    ).sma_indicator()

    df["EMA_10"] = EMAIndicator(
        close=df["Close"],
        window=10
    ).ema_indicator()

    # RSI
    df["RSI"] = RSIIndicator(
        close=df["Close"],
        window=14
    ).rsi()

    # MACD
    macd = MACD(close=df["Close"])

    df["MACD"] = macd.macd()

    # Rendement journalier
    df["Returns"] = df["Close"].pct_change()

    # Volatilité
    df["Volatility"] = (
        df["Returns"]
        .rolling(window=10)
        .std()
    )

    # Supprimer NaN
    df = df.dropna().reset_index(drop=True)

    return df

# ─────────────────────────────────────────────
# Téléchargement d'une paire
# ─────────────────────────────────────────────
def download_pair(pair, start=START_DATE, end=END_DATE):

    ticker = FOREX_PAIRS[pair]

    logger.info(f"Téléchargement {pair} ({ticker}) ...")

    df = yf.download(
        ticker,
        start=start,
        end=end,
        interval="1d",
        auto_adjust=True,
        progress=False
    )

    if df.empty:
        raise RuntimeError(f"Aucune donnée pour {pair}")

    # Supprimer MultiIndex si existe
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    # Nettoyage
    df.index.name = "Date"

    df = df.reset_index()[
        ["Date", "Open", "High", "Low", "Close", "Volume"]
    ]

    df = (
        df.dropna(subset=["Close"])
          .sort_values("Date")
          .reset_index(drop=True)
    )
    # Ajout des indicateurs techniques
    df = add_indicators(df)

    df["Pair"] = pair

    logger.info(
        f"{len(df)} lignes | "
        f"{df['Date'].min().date()} → "
        f"{df['Date'].max().date()}"
    )

    return df

# ─────────────────────────────────────────────
# Collection complète
# ─────────────────────────────────────────────
def run_data_collection():

    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

    logger.info("=" * 50)
    logger.info("LAB SESSION 2 — DATA COLLECTION")
    logger.info("=" * 50)

    datasets = {}

    for pair in FOREX_PAIRS:

        df = download_pair(pair)

        datasets[pair] = df
        # Afficher statistiques
        show_statistics(df, pair)

        # Afficher graphique
        plot_forex(df, pair)

        # Sauvegarde CSV
        path = RAW_DATA_DIR / f"{pair}.csv"

        df.to_csv(path, index=False)

        logger.info(f"Sauvegardé : {path}")

    # Résumé
    print("\n" + "=" * 50)
    print("RÉSUMÉ")
    print("=" * 50)

    for pair, df in datasets.items():

        print(
            f"{pair:8s} | "
            f"{len(df):5d} lignes | "
            f"Close min={df['Close'].min():.4f} | "
            f"max={df['Close'].max():.4f}"
        )

    print("=" * 50)

    return datasets
# ─────────────────────────────────────────────
# Statistiques descriptives
# ─────────────────────────────────────────────
def show_statistics(df, pair):

    print("\n" + "=" * 50)
    print(f"STATISTIQUES — {pair}")
    print("=" * 50)

    print(df["Close"].describe())

    print("\nValeurs manquantes :")
    print(df.isnull().sum())

    print("\nVolatilité moyenne :")
    print(df["Volatility"].mean())

# ─────────────────────────────────────────────
# Visualisation
# ─────────────────────────────────────────────
def plot_forex(df, pair):

    plt.figure(figsize=(14,6))

    plt.plot(
        df["Date"],
        df["Close"],
        label="Close Price",
        linewidth=2
    )

    plt.plot(
        df["Date"],
        df["SMA_10"],
        label="SMA 10",
        linestyle="--"
    )

    plt.plot(
        df["Date"],
        df["EMA_10"],
        label="EMA 10",
        linestyle=":"
    )

    plt.title(f"{pair} — Historical Prices")

    plt.xlabel("Date")

    plt.ylabel("Exchange Rate")

    plt.legend()

    plt.grid(True)

    plt.show()

# ─────────────────────────────────────────────
# Exécution
# ─────────────────────────────────────────────
datasets = run_data_collection()

In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║    LAB SESSION 2 — PHASE 2 : PREPROCESSING (Régression)     ║
║          Compatible Google Colab                            ║
║                                                              ║
║  Tâche   : Régression — prédire Close(t+1)                   ║
║  Scaler  : MinMaxScaler [0,1] sur Close uniquement           ║
║  Output  : X (samples, window, 1)  y (samples, 1)            ║
╚══════════════════════════════════════════════════════════════╝
"""

import logging
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

# ─────────────────────────────────────────────
# Dossiers Google Colab
# ─────────────────────────────────────────────
RAW_DATA_DIR       = Path("/content/data/raw")
PROCESSED_DATA_DIR = Path("/content/data/processed")
MODELS_DIR         = Path("/content/models")

# ─────────────────────────────────────────────
# Paramètres
# ─────────────────────────────────────────────
WINDOW_SIZE = 60
TRAIN_RATIO = 0.80

# ─────────────────────────────────────────────
# Logging
# ─────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────
# Chargement CSV
# ─────────────────────────────────────────────
def load_pair(pair):

    path = RAW_DATA_DIR / f"{pair}.csv"

    if not path.exists():
        raise FileNotFoundError(
            f"Aucun fichier trouvé : {path}"
        )

    df = pd.read_csv(path, parse_dates=["Date"])

    df = (
        df.sort_values("Date")
          .reset_index(drop=True)
    )

    logger.info(f"{pair} chargé : {len(df)} lignes")

    return df

# ─────────────────────────────────────────────
# Preprocessing d'une paire
# ─────────────────────────────────────────────
def preprocess_pair(
    pair,
    window=WINDOW_SIZE,
    train_ratio=TRAIN_RATIO
):

    df = load_pair(pair)

    # Série Close
    features = [
        "Close",
        "SMA_10",
        "EMA_10",
        "RSI",
        "MACD",
        "Returns",
        "Volatility"
    ]

    data_values = df[features].values
    close_raw = df["Close"].values.reshape(-1, 1)

    dates = df["Date"].values

    # ─────────────────────────────────────────
    # Split train brut
    # ─────────────────────────────────────────
    n_train = int(len(data_values) * train_ratio)

    # ─────────────────────────────────────────
    # Normalisation
    # ─────────────────────────────────────────
    scaler = MinMaxScaler()

    # Fit uniquement sur train
    scaler.fit(data_values[:n_train])

    scaled_data = scaler.transform(data_values)

    # ─────────────────────────────────────────
    # Création des séquences
    # ─────────────────────────────────────────
    X = []
    y = []
    y_dates = []

    for i in range(window, len(scaled_data)):

        X.append(scaled_data[i-window:i])

        y.append(scaled_data[i, 0])  # Close seulement

        y_dates.append(dates[i])

    # Conversion numpy
    X = np.array(X, dtype=np.float32)


    y = np.array(y, dtype=np.float32).reshape(-1, 1)

    y_dates = np.array(y_dates)

    # ─────────────────────────────────────────
    # Split chronologique
    # Train / Validation / Test
    # ─────────────────────────────────────────

    seq_train_end = n_train - window

    # Train + Validation
    X_train_full = X[:seq_train_end]
    y_train_full = y[:seq_train_end]

    dates_train_full = y_dates[:seq_train_end]

    # Test
    X_test = X[seq_train_end:]
    y_test = y[seq_train_end:]

    dates_test = y_dates[seq_train_end:]

    # ─────────────────────────────────────────
    # Validation split
    # 80% train — 20% validation
    # ─────────────────────────────────────────

    val_split = int(len(X_train_full) * 0.8)

    X_train = X_train_full[:val_split]
    y_train = y_train_full[:val_split]

    X_val = X_train_full[val_split:]
    y_val = y_train_full[val_split:]

    dates_train = dates_train_full[:val_split]
    dates_val   = dates_train_full[val_split:]

    logger.info(
        f"X_train={X_train.shape} | "
        f"X_val={X_val.shape} | "
        f"X_test={X_test.shape}"
    )

    logger.info(
        f"Train : {str(dates_train[0])[:10]} → "
        f"{str(dates_train[-1])[:10]}"
    )

    logger.info(
        f"Val   : {str(dates_val[0])[:10]} → "
        f"{str(dates_val[-1])[:10]}"
    )

    logger.info(
        f"Test  : {str(dates_test[0])[:10]} → "
        f"{str(dates_test[-1])[:10]}"
    )

    # ─────────────────────────────────────────
    # Création dossiers
    # ─────────────────────────────────────────
    PROCESSED_DATA_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    MODELS_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # ─────────────────────────────────────────
    # Sauvegarde
    # ─────────────────────────────────────────
    np.save(
        PROCESSED_DATA_DIR / f"{pair}_X_train.npy",
        X_train
    )

    np.save(
        PROCESSED_DATA_DIR / f"{pair}_X_test.npy",
        X_test
    )

    np.save(
        PROCESSED_DATA_DIR / f"{pair}_y_train.npy",
        y_train
    )

    np.save(
        PROCESSED_DATA_DIR / f"{pair}_y_test.npy",
        y_test
    )

    np.save(
        PROCESSED_DATA_DIR / f"{pair}_X_val.npy",
        X_val
    )

    np.save(
        PROCESSED_DATA_DIR / f"{pair}_y_val.npy",
        y_val
    )

    joblib.dump(
        scaler,
        MODELS_DIR / f"{pair}_scaler.pkl"
    )

    logger.info(f"Préprocessing terminé : {pair}")

    return {
        "pair": pair,

        "X_train": X_train,
        "y_train": y_train,

        "X_test": X_test,
        "y_test": y_test,

        "dates_train": dates_train,
        "dates_test": dates_test,

        "scaler": scaler,

        "close_raw": close_raw,

        "dates_all": dates,

        "window": window,

        "n_train": n_train,

        "X_val": X_val,
        "y_val": y_val,

        "dates_val": dates_val,
    }

# ─────────────────────────────────────────────
# Preprocessing global
# ─────────────────────────────────────────────
def run_preprocessing(pairs=None):

    if pairs is None:

        pairs = [
            "EURUSD",
            "EURMAD",
            "USDMAD"
        ]

    logger.info("=" * 55)
    logger.info("PREPROCESSING — FOREX")
    logger.info("=" * 55)

    all_data = {}

    for pair in pairs:

        logger.info(f"\n── {pair} ─────────────")

        all_data[pair] = preprocess_pair(pair)

    # ─────────────────────────────────────────
    # Résumé
    # ─────────────────────────────────────────
    print("\n" + "=" * 55)

    print("RÉSUMÉ PREPROCESSING")

    print("=" * 55)

    for pair, d in all_data.items():

        print(
            f"{pair} | "
            f"train={d['X_train'].shape} | "
            f"val={d['X_val'].shape} | "
            f"test={d['X_test'].shape}"
        )

    print("=" * 55)

    return all_data

# ─────────────────────────────────────────────
# Exécution
# ─────────────────────────────────────────────
data = run_preprocessing()